# Notebook 1 – Introduction to Feature Engineering

## Learning objectives
By the end of this notebook, you should be able to explain what a feature is, distinguish features from targets, describe the feature-engineering workflow, and recognize poor features and data leakage.

## 1. What is a Feature?

A **feature** is an input column that provides information to a Machine Learning model. A feature can describe a customer, product, transaction, or event.

### Example
In a customer churn dataset, `tenure`, `MonthlyCharges`, and `Contract` can be features because they describe a customer's relationship with a service.

### Code – Create a small example dataset

In [1]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "tenure_months": [2, 18, 36, 7, 50],
    "monthly_charges": [70.0, 55.0, 90.0, 45.0, 80.0],
    "contract": ["Month-to-month", "One year", "Two year", "Month-to-month", "Two year"],
    "churn": ["Yes", "No", "No", "Yes", "No"]
})

customers

,customer_id,tenure_months,monthly_charges,contract,churn
0,101,2,70.0,Month-to-month,Yes
1,102,18,55.0,One year,No
2,103,36,90.0,Two year,No
3,104,7,45.0,Month-to-month,Yes
4,105,50,80.0,Two year,No


### Output explanation
Each row represents one customer. The descriptive columns are possible input features, while `churn` is the outcome we want to predict.

## 2. Feature vs Variable

A **variable** is any measured or recorded property in a dataset. A **feature** is a variable selected or prepared for use as model input. A variable does not automatically become a useful feature.

### Example
`customer_id` is a variable, but it usually should not be used as a predictive feature because its number may only identify a record.

### Code – Inspect variables and candidate features

In [2]:
print("All variables:", customers.columns.tolist())

candidate_features = [
    column for column in customers.columns
    if column not in ["customer_id", "churn"]
]
print("Candidate model features:", candidate_features)

All variables: ['customer_id', 'tenure_months', 'monthly_charges', 'contract', 'churn']
Candidate model features: ['tenure_months', 'monthly_charges', 'contract']


### Output explanation
The ID and target are excluded from the candidate input list. This is an initial decision; each candidate still needs to be checked for relevance and leakage.

## 3. Feature vs Target

- **Features (X):** information supplied to the model.
- **Target (y):** the value the model is expected to predict.

### Example
For churn prediction, customer details are `X` and the churn label is `y`.

In [3]:
X = customers[["tenure_months", "monthly_charges", "contract"]]
y = customers["churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget values:")
print(y.value_counts())

X shape: (5, 3)
y shape: (5,)

Target values:
churn
No     3
Yes    2
Name: count, dtype: int64


### Output explanation
`X` contains three input columns and `y` contains one outcome column. The target must not accidentally be included among the input features.

## 4. What is Feature Engineering?

**Feature Engineering** is the process of creating, transforming, or preparing input features so that they represent useful information for a Machine Learning problem.

### Example
A customer’s `tenure_months` can be converted into `tenure_years`. The new column expresses the same information in a different, potentially more interpretable scale.

In [4]:
customers["tenure_years"] = customers["tenure_months"] / 12
customers[["tenure_months", "tenure_years"]]

,tenure_months,tenure_years
0,2,0.166667
1,18,1.500000
2,36,3.000000
3,7,0.583333
4,50,4.166667


### Output explanation
The new feature is derived from `tenure_months`. It may make customer duration easier to interpret, although it does not add new underlying information.

## 5. Why Feature Engineering?

Feature engineering can:
- expose patterns that are hidden in raw columns;
- represent business concepts more clearly;
- make relationships easier for a model to learn;
- reduce noise when irrelevant information is removed.

### Example
For retail, `quantity × unit_price` can represent line-item sales value. A model may find this more directly useful than interpreting the two columns independently.

## 6. Importance of Domain Knowledge

Domain knowledge means understanding the real-world process represented by the data. It helps decide whether a calculation is meaningful and whether it would be available at prediction time.

### Example
In telecom, a high number of support calls may indicate customer dissatisfaction. In banking, recent missed payments may be relevant to credit risk.

## 7. Good Features vs Bad Features

A good feature is relevant, reliable, available at prediction time, and understandable in context. A bad feature may be irrelevant, mostly unique, duplicated, unstable, or derived from information that would not yet be known.

### Example
`tenure_months` may be useful for churn prediction. A randomly generated row number is unlikely to carry genuine customer behaviour information.

In [5]:
feature_review = pd.DataFrame({
    "feature": ["tenure_months", "monthly_charges", "customer_id", "churn"],
    "initial_role": ["Candidate input", "Candidate input", "Identifier", "Target"],
    "reason": [
        "May describe relationship length",
        "May describe service cost",
        "Usually only identifies a row",
        "Value to predict"
    ]
})
feature_review

,feature,initial_role,reason
0,tenure_months,Candidate input,May describe relationship length
1,monthly_charges,Candidate input,May describe service cost
2,customer_id,Identifier,Usually only identifies a row
3,churn,Target,Value to predict


### Output explanation
This is an initial review, not an automatic final selection. Validate usefulness with domain knowledge and training-only evaluation.

## 8. Feature Engineering Workflow

A practical workflow is:

1. Understand the business problem and target.
2. Inspect available columns and data types.
3. Identify gaps in the information represented.
4. Create or transform candidate features.
5. Check quality, relevance, and leakage risk.
6. Compare candidate features using training data.
7. Retain useful features and document decisions.

### Example
For churn prediction, inspect customer start date, calculate tenure as of the prediction date, and verify that the calculation does not use a future cancellation date.

## 9. Feature Creation

Feature creation combines existing information or extracts a new representation.

### Example
`total_spend = monthly_charges × tenure_months` is a simple illustrative estimate. It should not be treated as actual billed spend unless the business definition supports it.

In [6]:
customers["estimated_spend"] = (
    customers["monthly_charges"] * customers["tenure_months"]
)
customers[["monthly_charges", "tenure_months", "estimated_spend"]]

,monthly_charges,tenure_months,estimated_spend
0,70.0,2,140.0
1,55.0,18,990.0
2,90.0,36,3240.0
3,45.0,7,315.0
4,80.0,50,4000.0


### Output explanation
The calculation creates an estimate from two existing columns. Its business meaning depends on whether monthly charges were constant throughout the customer's tenure.

## 10. Feature Transformation

Transformation changes the scale or representation of an existing feature, for example using a logarithm, standardization, or encoding.

### Example
A right-skewed transaction amount may be log-transformed to reduce the influence of very large values.

In [7]:
import numpy as np

amounts = pd.Series([10, 20, 35, 100, 1500], name="amount")
transformed = np.log1p(amounts)
pd.DataFrame({"amount": amounts, "log1p_amount": transformed})

,amount,log1p_amount
0,10,2.397895
1,20,3.044522
2,35,3.583519
3,100,4.615121
4,1500,7.313887


### Output explanation
`log1p` computes `log(1 + x)` and safely handles zero. It changes the scale; it does not remove the need to inspect outliers or understand the data.

## 11. Feature Selection

Feature selection chooses a subset of available features. It can remove irrelevant or redundant columns and simplify a model.

### Example
If two columns contain almost identical information, one may be removed after checking their meaning and model impact.

## 12. Feature Extraction

Feature extraction creates a new representation from one or more existing variables. Unlike simple selection, it can combine or compress information.

### Example
A date such as `2026-11-05` can be represented using year, month, day, and weekday.

In [8]:
dates = pd.Series(pd.to_datetime(["2026-11-05", "2026-11-07"]))
date_features = pd.DataFrame({
    "date": dates,
    "year": dates.dt.year,
    "month": dates.dt.month,
    "day_of_week": dates.dt.day_name()
})
date_features

,date,year,month,day_of_week
0,2026-11-05,2026,11,Thursday
1,2026-11-07,2026,11,Saturday


### Output explanation
The original date is converted into several meaningful components. Which components are useful depends on the problem and possible seasonal patterns.

## 13. Feature Leakage

**Feature leakage** occurs when a model receives information during training that would not legitimately be available when making a real prediction. Leakage can produce unrealistically strong evaluation results.

### Example
If predicting whether a customer will churn next month, a column recording the actual cancellation date may reveal the answer. It must not be used as a prediction-time feature.

In [9]:
leakage_example = pd.DataFrame({
    "customer_id": [201, 202, 203],
    "monthly_charges": [60, 80, 45],
    "cancellation_date": ["2026-10-10", None, "2026-10-25"],
    "churn": ["Yes", "No", "Yes"]
})
leakage_example

,customer_id,monthly_charges,cancellation_date,churn
0,201,60,2026-10-10,Yes
1,202,80,NaN,No
2,203,45,2026-10-25,Yes


### Output explanation
`cancellation_date` is known only after the cancellation event. Including it in a model intended to predict future churn would leak outcome information.

## 14. Role of Feature Engineering in the ML Pipeline

Feature engineering sits between data preparation and model training. It turns cleaned raw columns into meaningful model inputs. It should be fitted using training data only when a transformation learns parameters, then applied consistently to validation, test, and production data.

**Pipeline:** Raw data → Data cleaning → Feature engineering → Feature selection → Model training → Evaluation → Deployment